# Aula 12 · Lab 2 — OpenSearch (Vector DB) + LangFuse (Monitoramento)

> **Objetivo:** Configurar e integrar OpenSearch como backend de busca vetorial e BM25,  
> e LangFuse para rastreabilidade completa das queries em produção.

## Pré-requisito

Execute o Docker Compose do Lab 4 antes de rodar este notebook:  
```bash
docker compose up -d opensearch langfuse-server langfuse-db
```

---
**Referências:** OpenSearch Docs (2024) · LangFuse Production Docs (2024)

In [ ]:
!pip install -q opensearch-py langfuse openai sentence-transformers
print("✅ Pronto")

In [ ]:
import os

# OpenSearch
OS_HOST  = os.environ.get("OPENSEARCH_HOST", "localhost")
OS_PORT  = int(os.environ.get("OPENSEARCH_PORT", 9200))
OS_USER  = os.environ.get("OPENSEARCH_USER", "admin")
OS_PASS  = os.environ.get("OPENSEARCH_PASS", "admin")
OS_INDEX = "rag_projeto_final"

# LangFuse
os.environ["LANGFUSE_PUBLIC_KEY"] = "pk-lf-..."   # ← SUBSTITUA
os.environ["LANGFUSE_SECRET_KEY"] = "sk-lf-..."   # ← SUBSTITUA
os.environ["LANGFUSE_HOST"]       = "http://localhost:3000"  # LangFuse local

# OpenAI
os.environ["OPENAI_API_KEY"] = "sk-..."  # ← SUBSTITUA

print("✅ Configurações carregadas")

In [ ]:
# ============================================================
# PASSO 1: Verificar conexão com OpenSearch
# ============================================================

from opensearchpy import OpenSearch, RequestsHttpConnection
import warnings
warnings.filterwarnings("ignore")

try:
    os_client = OpenSearch(
        hosts=[{"host": OS_HOST, "port": OS_PORT}],
        http_auth=(OS_USER, OS_PASS),
        use_ssl=False,
        verify_certs=False,
        connection_class=RequestsHttpConnection,
        timeout=30,
    )
    info = os_client.info()
    print(f"✅ OpenSearch {info['version']['number']} conectado")
    print(f"   Cluster: {info['cluster_name']}")

    # Verifica saúde do cluster
    saude = os_client.cluster.health()
    print(f"   Status: {saude['status']} | Nós: {saude['number_of_nodes']}")

except Exception as e:
    print(f"❌ OpenSearch não disponível: {e}")
    print("   Execute: docker compose up -d opensearch")
    os_client = None

In [ ]:
# ============================================================
# PASSO 2: Criar índice com mapeamento híbrido
# ============================================================

MAPEAMENTO = {
    "settings": {
        "index": {
            "knn": True,
            "knn.algo_param.ef_search": 512,
            "number_of_shards": 1,    # Dev: 1 shard. Prod: calcule por volume
            "number_of_replicas": 0   # Dev: 0. Prod: >= 1 para HA
        }
    },
    "mappings": {
        "properties": {
            "embedding": {
                "type": "knn_vector",
                "dimension": 1536,
                "method": {
                    "name": "hnsw",
                    "space_type": "cosinesimil",
                    "engine": "lucene",
                    "parameters": {"m": 16, "ef_construction": 256}
                }
            },
            "texto":     {"type": "text", "analyzer": "portuguese"},
            "titulo":    {"type": "text", "analyzer": "portuguese"},
            "chunk_id":  {"type": "keyword"},
            "doc_id":    {"type": "keyword"},
            "tipo":      {"type": "keyword"},
            "fonte":     {"type": "text"},
            "chunk_index": {"type": "integer"},
        }
    }
}

if os_client:
    # Remove índice existente (para recriar limpo)
    if os_client.indices.exists(index=OS_INDEX):
        os_client.indices.delete(index=OS_INDEX)
        print(f"🗑️  Índice '{OS_INDEX}' removido")

    os_client.indices.create(index=OS_INDEX, body=MAPEAMENTO)
    print(f"✅ Índice '{OS_INDEX}' criado")

    # Inspeciona o mapeamento criado
    mapping = os_client.indices.get_mapping(index=OS_INDEX)
    campos = list(mapping[OS_INDEX]["mappings"]["properties"].keys())
    print(f"   Campos mapeados: {campos}")

In [ ]:
# ============================================================
# PASSO 3: Indexar chunks do Lab 1 no OpenSearch
# ============================================================
# Requer: todos_chunks e gerar_embedding() do Lab 1
# Se estiver executando este lab isoladamente, defina um corpus mock:

from openai import OpenAI
from opensearchpy.helpers import bulk
import time

openai_client = OpenAI()

# Corpus mock (substitua por todos_chunks do Lab 1)
CHUNKS_PARA_INDEXAR = [
    {"chunk_id": "doc001_001", "doc_id": "doc001", "titulo": "LGPD Art. 5",
     "texto": "Art. 5º Dado pessoal sensível: dado sobre origem racial, saúde, biometria.",
     "tipo": "legislacao", "fonte": "Lei 13.709/2018", "chunk_index": 0},
    {"chunk_id": "doc002_001", "doc_id": "doc002", "titulo": "ECA Art. 2",
     "texto": "Art. 2º Adolescente: pessoa entre doze e dezoito anos.",
     "tipo": "legislacao", "fonte": "Lei 8.069/1990", "chunk_index": 0},
]


def indexar_com_embedding(cliente, indice, chunks, batch_size=20):
    """Indexa chunks gerando embeddings em lote."""
    total = 0
    for i in range(0, len(chunks), batch_size):
        lote = chunks[i : i + batch_size]
        textos = [c["texto"][:512] for c in lote]

        # Gera embeddings
        resp = openai_client.embeddings.create(input=textos, model="text-embedding-3-small")
        embeddings = [item.embedding for item in resp.data]

        # Prepara ações de bulk
        acoes = [
            {"_index": indice, "_id": c["chunk_id"], "_source": {**c, "embedding": emb}}
            for c, emb in zip(lote, embeddings)
        ]

        sucesso, _ = bulk(cliente, acoes, raise_on_error=False)
        total += sucesso
        print(f"   Lote {i//batch_size + 1}: {sucesso}/{len(lote)} indexados")

    return total


if os_client:
    print("🔄 Indexando chunks...")
    total = indexar_com_embedding(os_client, OS_INDEX, CHUNKS_PARA_INDEXAR)
    os_client.indices.refresh(index=OS_INDEX)
    count = os_client.cat.count(index=OS_INDEX, format="json")[0]["count"]
    print(f"\n✅ {total} chunks indexados | Total no índice: {count}")

In [ ]:
# ============================================================
# PASSO 4: Configurar LangFuse e criar primeiro trace
# ============================================================

from langfuse import Langfuse
import uuid

try:
    langfuse = Langfuse(
        public_key=os.environ["LANGFUSE_PUBLIC_KEY"],
        secret_key=os.environ["LANGFUSE_SECRET_KEY"],
        host=os.environ["LANGFUSE_HOST"]
    )
    # Testa autenticação
    langfuse.auth_check()
    print("✅ LangFuse conectado")
    print(f"   Host: {os.environ['LANGFUSE_HOST']}")
    LANGFUSE_OK = True

except Exception as e:
    print(f"⚠️  LangFuse não disponível: {e}")
    print("   Verifique as credenciais e o host")
    LANGFUSE_OK = False
    langfuse = None

In [ ]:
# ============================================================
# PASSO 5: Pipeline RAG com trace LangFuse completo
# ============================================================

def pipeline_com_trace(
    pergunta: str,
    usuario_id: str = "lab2_teste"
):
    """
    Pipeline RAG com rastreabilidade completa no LangFuse.
    Registra: embedding, retrieval, geração, scores e latência.
    """
    session_id = str(uuid.uuid4())
    inicio_total = time.time()

    # Cria trace raiz
    trace = langfuse.trace(
        name="rag_query",
        input={"pergunta": pergunta},
        user_id=usuario_id,
        session_id=session_id,
        metadata={"opensearch_index": OS_INDEX}
    ) if LANGFUSE_OK else None

    resultado = {}

    try:
        # --- SPAN: Embedding ---
        t0 = time.time()
        span_emb = trace.span(name="embedding") if trace else None
        resp_emb = openai_client.embeddings.create(input=[pergunta], model="text-embedding-3-small")
        embedding = resp_emb.data[0].embedding
        latencia_emb = (time.time() - t0) * 1000
        if span_emb: span_emb.end(output={"latencia_ms": round(latencia_emb, 1)})

        # --- SPAN: Retrieval OpenSearch ---
        t0 = time.time()
        span_ret = trace.span(name="retrieval_opensearch") if trace else None

        if os_client:
            query_body = {"size": 5, "query": {"knn": {"embedding": {"vector": embedding, "k": 5}}}}
            hits = os_client.search(index=OS_INDEX, body=query_body)["hits"]["hits"]
            chunks_ret = [{"titulo": h["_source"]["titulo"], "texto": h["_source"]["texto"], "score": h["_score"]} for h in hits]
        else:
            chunks_ret = [{"titulo": "Mock", "texto": "Chunk mock para teste", "score": 0.9}]

        latencia_ret = (time.time() - t0) * 1000
        if span_ret: span_ret.end(output={"chunks": len(chunks_ret), "latencia_ms": round(latencia_ret, 1)})

        # --- SPAN: Geração LLM ---
        t0 = time.time()
        contexto = "\n\n".join([f"[{c['titulo']}]\n{c['texto']}" for c in chunks_ret])

        resp_llm = openai_client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": "Responda com base nos documentos. Cite a fonte."},
                {"role": "user", "content": f"Contexto:\n{contexto}\n\nPergunta: {pergunta}"}
            ],
            temperature=0.0, max_tokens=512
        )
        resposta = resp_llm.choices[0].message.content
        latencia_llm = (time.time() - t0) * 1000

        # Registra geração no LangFuse
        if trace:
            trace.generation(
                name="gpt_completion",
                model="gpt-4o-mini",
                model_parameters={"temperature": 0.0},
                input=[{"role": "user", "content": pergunta}],
                output=resposta,
                usage={"input": resp_llm.usage.prompt_tokens,
                       "output": resp_llm.usage.completion_tokens,
                       "total": resp_llm.usage.total_tokens}
            )

        latencia_total = (time.time() - inicio_total) * 1000

        # Scores de qualidade (substitua por RAGAS real)
        if trace:
            langfuse.score(trace_id=trace.id, name="faithfulness", value=0.9)
            langfuse.score(trace_id=trace.id, name="latencia_ms", value=round(latencia_total, 0))

        if trace: trace.update(output={"resposta": resposta[:200]})

        resultado = {
            "resposta": resposta,
            "chunks": len(chunks_ret),
            "latencia_total_ms": round(latencia_total, 1),
            "latencia_emb_ms": round(latencia_emb, 1),
            "latencia_ret_ms": round(latencia_ret, 1),
            "latencia_llm_ms": round(latencia_llm, 1),
            "tokens": resp_llm.usage.total_tokens,
            "trace_id": trace.id if trace else None,
        }

    finally:
        if langfuse: langfuse.flush()

    return resultado


# Teste
r = pipeline_com_trace("O que é dado pessoal sensível?")
print("✅ Pipeline com trace executado")
print(f"\nResposta: {r['resposta'][:150]}...")
print(f"\nLatências:")
print(f"  Embedding:  {r['latencia_emb_ms']}ms")
print(f"  Retrieval:  {r['latencia_ret_ms']}ms")
print(f"  LLM:        {r['latencia_llm_ms']}ms")
print(f"  Total:      {r['latencia_total_ms']}ms")
if r.get('trace_id'):
    print(f"\n🔗 Trace: {os.environ['LANGFUSE_HOST']}/trace/{r['trace_id']}")

In [ ]:
# ============================================================
# PASSO 6: Configurar alertas de degradação no LangFuse
# ============================================================
# Na interface web do LangFuse, configure alertas em:
# Settings > Alerts > New Alert
#
# Alerta recomendado 1: Faithfulness < 0.7
#   - Métrica: Score 'faithfulness'
#   - Condição: média < 0.7 nos últimos 50 traces
#   - Ação: Email ou webhook
#
# Alerta recomendado 2: Latência P95 > 5000ms
#   - Métrica: latência de observação
#   - Condição: P95 > 5000ms na última hora
#
# Programaticamente, você pode listar traces recentes para análise:

if LANGFUSE_OK:
    try:
        traces = langfuse.fetch_traces(limit=10)
        print(f"✅ {len(traces.data)} traces recentes recuperados")
        for t in traces.data[:3]:
            print(f"   [{t.id[:8]}] {t.name} | {t.created_at}")
    except Exception as e:
        print(f"Erro ao buscar traces: {e}")
else:
    print("⚠️  LangFuse offline — configure as credenciais para ver traces")

print("\n📋 Configuração de alertas via interface web:")
print(f"   {os.environ['LANGFUSE_HOST']}/settings/alerts")